# Gen AI Engineering — A Minimal LLM Service Skeleton

**Category:** Gen AI Engineering · Chip Huyen's AI Engineering, AI Magazine, OpenClaw on AWS.

**What this notebook does.** Builds the *shape* of a production LLM service in a single notebook: a router that picks between a cheap and expensive model based on input characteristics, prompt templates versioned in code, an offline eval harness, and a structured-log emitter. There's no actual model call here (so the notebook is reproducible and key-free) — the 'models' are stubs you'd swap for OpenAI / Anthropic / Claude.

**The point.** Make the inside of a real service legible. This is the architecture a CTO is hiring for.

## 1. Prompt templates — versioned, in code

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Prompt:
    id: str
    version: str
    template: str
    def render(self, **kw): return self.template.format(**kw)

PROMPTS = {
    'classify_v1': Prompt(
        id='classify', version='v1',
        template='Classify the sentiment of: {text}\nAnswer one of: positive, negative, neutral.'
    ),
    'classify_v2': Prompt(
        id='classify', version='v2',
        template=('You are a careful sentiment classifier. For the text below, output ONLY '
                  'one of: positive, negative, neutral.\n\nText: {text}\nAnswer:')
    ),
}
print(PROMPTS['classify_v2'].render(text='I loved the service.'))


## 2. Model stubs — cheap vs expensive

In [ ]:
import random
random.seed(7)

def cheap_model(prompt: str):
    # simulate: fast, often correct on short input, occasionally noisy
    lower = prompt.lower()
    if 'loved' in lower or 'great' in lower or 'excellent' in lower: return 'positive'
    if 'hated' in lower or 'terrible' in lower or 'awful' in lower:  return 'negative'
    return random.choice(['positive', 'negative', 'neutral'])

def expensive_model(prompt: str):
    # simulate: more accurate but slower / costlier
    lower = prompt.lower()
    pos = any(w in lower for w in ['loved','great','excellent','enjoyed','recommended'])
    neg = any(w in lower for w in ['hated','terrible','awful','disappointed','poor'])
    if pos and not neg: return 'positive'
    if neg and not pos: return 'negative'
    return 'neutral'

MODEL_COSTS = {'cheap': 0.0002, 'expensive': 0.002}  # $ per request
MODEL_LATENCY = {'cheap': 0.05, 'expensive': 0.4}    # seconds


## 3. Router — input-length-aware tiering

In [ ]:
def route(text):
    if len(text) < 40: return 'cheap'
    return 'expensive'

MODELS = {'cheap': cheap_model, 'expensive': expensive_model}


## 4. Structured logging — what production observability needs

In [ ]:
import json, time, uuid
LOGS = []
def emit_log(event):
    LOGS.append(event)

def serve(text, prompt_key='classify_v2'):
    req_id = str(uuid.uuid4())[:8]
    tier = route(text)
    p = PROMPTS[prompt_key].render(text=text)
    t0 = time.time()
    out = MODELS[tier](p)
    dt = time.time() - t0
    emit_log({
        'req_id': req_id, 'tier': tier,
        'prompt_id': PROMPTS[prompt_key].id,
        'prompt_version': PROMPTS[prompt_key].version,
        'input_chars': len(text),
        'output': out,
        'latency_s': round(dt, 3),
        'cost_usd': MODEL_COSTS[tier],
    })
    return out

print(serve('I loved the service.'))
print(serve('The product was decent but the support team was unresponsive and I had to call three times.'))
print(json.dumps(LOGS[-1], indent=2))


## 5. Offline eval harness — versioned against the prompt

In [ ]:
EVAL = [
    ('I loved it, ten out of ten.', 'positive'),
    ('Awful experience, would not recommend.', 'negative'),
    ('It was okay, nothing special.', 'neutral'),
    ('The team was great and the product excellent.', 'positive'),
    ('Disappointed with poor support.', 'negative'),
    ('Average product, average service.', 'neutral'),
]

def eval_prompt(prompt_key):
    correct = 0
    for text, gold in EVAL:
        pred = serve(text, prompt_key=prompt_key)
        correct += (pred == gold)
    return correct / len(EVAL)

for key in ['classify_v1', 'classify_v2']:
    LOGS.clear()
    acc = eval_prompt(key)
    total_cost = sum(L['cost_usd'] for L in LOGS)
    avg_lat   = sum(L['latency_s'] for L in LOGS) / len(LOGS)
    print(f'{key}: accuracy {acc:.0%} | cost/req ${total_cost/len(EVAL):.4f} | avg latency {avg_lat:.2f}s')


## 6. Cost projection — the slide a CFO actually reads

In [ ]:
REQS_PER_DAY = 100_000
SHARE_EXPENSIVE = 0.4  # from your router stats
daily = REQS_PER_DAY * (
    (1 - SHARE_EXPENSIVE) * MODEL_COSTS['cheap'] +
    SHARE_EXPENSIVE * MODEL_COSTS['expensive']
)
print(f'Projected daily cost: ${daily:,.2f}')
print(f'Projected monthly:    ${daily*30:,.2f}')


## My take

What this notebook tries to make obvious: a production LLM service is *a service*, not a model. The model is a function call in the middle. The interesting code is the layer around it:

- **Prompts are code.** They live in version control, they have IDs and versions, the eval harness binds an eval score to a specific prompt version. If your prompts live in a Notion doc, you're going to ship a regression you can't reproduce.
- **Routing is the cheapest 30% cost cut you'll ever ship.** Send the short, easy queries to the cheap model. Send the hard ones to the expensive one. The router itself is twenty lines.
- **Structured logs from day one.** If your logs aren't JSON with `prompt_version`, `model`, `latency`, `cost`, and `input_chars` per request, you cannot answer the 'why did quality drop' question that will land on your desk in month three.
- **The eval harness is the most under-built thing on most teams.** It's also the highest-leverage. Build it before the second model integration.

Chip Huyen's *AI Engineering* book is the best single read on this material. The 80% of the work I see candidates underestimate is in this notebook.